# Part 1 · Dissociated melanoma → agent input

Use the public 10x melanoma example, a synthetic melanoma fixture, or one of your own annotated biopsies. Spatial coordinates are **not** inferred from RNA. Run cells in order. No data are uploaded automatically.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util

# In Colab, the public GitHub version must already contain this release.
INSTALL_PACKAGE = True
REPO_URL = "https://github.com/hylixinsights/VirtualTissue-scRNAseq.git"
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((d for d in candidates if (d / "pyproject.toml").exists() and (d / "src/vtsc").exists()), None)
if ROOT is None:
    ROOT = Path("/content/VirtualTissue-scRNAseq") if IN_COLAB else Path.home() / "VirtualTissue-scRNAseq"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
if INSTALL_PACKAGE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[notebooks]"], check=True)
sys.path.insert(0, str(ROOT / "src"))
print("Code:", ROOT)
# Leave WORK below local, or explicitly mount Drive and choose a Drive path.
WORK = Path(os.environ.get("VTSC_NOTEBOOK_WORK", str(ROOT / "outputs/notebook_melanoma")))
WORK.mkdir(parents=True, exist_ok=True)
print("Files:", WORK)


## Choose the biopsy

The default downloads the pinned CC BY 4.0 dissociated melanoma matrix (~57 MB). `synthetic` runs offline. For `own`, supply reviewed cell types and raw counts. See `docs/02_your_own_data.md`. Use a new WORK directory for a new experiment.

In [ ]:
DATA_SOURCE = "melanoma"  # melanoma | synthetic | own
OWN_INPUT = Path("/path/to/biopsy.h5ad")  # also Cell Ranger H5 or 10x MTX directory
OWN_METADATA = None
COUNT_LAYER = None  # 'counts' if H5AD.X is normalized
CELL_TYPE_COLUMN = 'cell_type'
SAMPLE_ID = None  # required when input contains more than one sample
MAX_CELLS = 400
SEED = 21
BUNDLE = WORK / 'input.zip'


In [ ]:
from vtsc.tissue_inputs import download_melanoma, synthetic_melanoma, prepare_tissue, read_tissue_bundle
if BUNDLE.exists():
    raise FileExistsError('Reuse this bundle in Part 2, or choose a new WORK directory.')
if DATA_SOURCE == 'melanoma':
    source = download_melanoma(WORK / 'melanoma10x')['input']
    info = prepare_tissue(source, BUNDLE, recipe='melanoma-markers', max_cells=MAX_CELLS, seed=SEED)
elif DATA_SOURCE == 'synthetic':
    source = WORK / 'synthetic_melanoma.h5ad'
    synthetic_melanoma(source, cells=300, seed=SEED)
    info = prepare_tissue(source, BUNDLE, max_cells=MAX_CELLS, seed=SEED, source_license='MIT; generated fixture')
elif DATA_SOURCE == 'own':
    info = prepare_tissue(OWN_INPUT, BUNDLE, metadata=OWN_METADATA, layer=COUNT_LAYER,
                          cell_type_column=CELL_TYPE_COLUMN, sample_id=SAMPLE_ID,
                          max_cells=MAX_CELLS, seed=SEED)
else:
    raise ValueError('Unknown DATA_SOURCE')
info


## Review composition before simulation

All selected lineages remain in the sample. The example marker recipe is provisional and may leave Unknown cells. Review annotations before making biological interpretations. Uniform subsampling does not preserve original spatial neighborhoods.

In [ ]:
import pandas as pd
payload = read_tissue_bundle(BUNDLE)
display(pd.DataFrame({'full': info['full_composition'], 'selected': info['sampled_composition']}).fillna(0).astype(int))
print('Bundle:', BUNDLE)
print('Annotation:', info['annotation'])
print('Geometry:', info['coordinates'])


Continue with Parts 2–3 using the same WORK directory. The bundle contains selected cell IDs, labels and program scores, not the full expression matrix.